### Load and Split Dataset

In [5]:
import pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Load your CSV
df = pd.read_csv("nav_dataset.csv")

# Split: 90% for training, 10% for validation (checking during training)
train_df, val_df = train_test_split(df, test_size=0.1, random_state=42)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df.reset_index(drop=True))

print(f"Training examples: {len(train_dataset)}")
print(f"Validation examples: {len(val_dataset)}")

Training examples: 218
Validation examples: 25


### Tokenize Data

In [6]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("t5-small")

def preprocess(examples):
    # Encode the inputs (navigation JSON strings)
    model_inputs = tokenizer(
        examples["input"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )
    # Encode the targets (natural language instructions)
    labels = tokenizer(
        examples["target"],
        max_length=64,
        truncation=True,
        padding="max_length"
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(preprocess, batched=True)
tokenized_val = val_dataset.map(preprocess, batched=True)

Map: 100%|██████████| 25/25 [00:00<00:00, 3674.31 examples/s]


### Model Loading and Configuration

In [10]:
from transformers import T5ForConditionalGeneration, TrainingArguments, Trainer

model = T5ForConditionalGeneration.from_pretrained("t5-small")

training_args = TrainingArguments(
    output_dir="./nav_t5_model_v1",   # Where to save checkpoints
    num_train_epochs=10,              # How many times to loop through all data
    per_device_train_batch_size=16,   # Examples processed together per step
    per_device_eval_batch_size=16,
    warmup_steps=50,                  # Gradual learning rate warm-up
    weight_decay=0.01,                # Regularisation to prevent overfitting
    logging_dir="./logs",
    logging_steps=10,
    eval_strategy="epoch",      # Evaluate after every epoch
    save_strategy="epoch",
    load_best_model_at_end=True,      # Keep the best checkpoint
    fp16=False,                        # Faster training on GPU (16-bit floats)
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=tokenizer,
)

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 7870.14it/s]
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


### Train the model

In [ ]:
trainer.train()

c:\Users\SankhaAmbeypitiya\Projects\Other\fyp\feedback\nlp-tryout\venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


### Save Model

In [ ]:
trainer.save_model("./nav_t5_final_v1")
tokenizer.save_pretrained("./nav_t5_final_v1")
print("Model saved!")